# Feature Selection Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forge-features/forge/blob/main/notebooks/03_feature_selection.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/forge-features/forge/main?labpath=notebooks/03_feature_selection.ipynb)

This notebook demonstrates Forge's feature selection capabilities.

## What you'll learn

1. Statistical feature selection (chi-square, ANOVA, mutual information)
2. Importance-based selection (tree-based importance)
3. Correlation-based selection
4. Variance-based selection
5. SHAP-based selection

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

np.random.seed(42)

# Create a dataset with known informative features
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=5,
    n_redundant=5,
    n_repeated=0,
    n_clusters_per_class=2,
    random_state=42
)

# Convert to DataFrame
feature_names = [f'feature_{i}' for i in range(20)]
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name='target')

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")

## Statistical Feature Selection

Use statistical tests to select features:

In [ ]:
from forge.selectors import StatisticalSelector

# ANOVA F-test (for classification)
selector = StatisticalSelector(
    method='anova',
    k=10  # Select top 10 features
)

X_selected = selector.fit_transform(X, y)

print(f"Selected {X_selected.shape[1]} features using ANOVA:")
print(list(X_selected.columns))

# Get feature scores
scores = selector.get_scores()
print("\nFeature scores:")
print(scores.sort_values(ascending=False).head(10))

In [ ]:
# Mutual Information
mi_selector = StatisticalSelector(
    method='mutual_info_classif',
    k=10
)

X_mi = mi_selector.fit_transform(X, y)
print(f"Selected features using Mutual Information:")
print(list(X_mi.columns))

## Importance-Based Selection

Use tree-based models to compute feature importance:

In [ ]:
from forge.selectors import ImportanceSelector

# Select features using Random Forest importance
imp_selector = ImportanceSelector(
    k=10,
    estimator='random_forest'
)

X_imp = imp_selector.fit_transform(X, y)

print(f"Selected {X_imp.shape[1]} features using importance:")
print(list(X_imp.columns))

# Get importance scores
importance = imp_selector.get_importance()
print("\nFeature importance:")
print(importance.sort_values(ascending=False).head(10))

## Correlation-Based Selection

Remove highly correlated features to reduce redundancy:

In [ ]:
from forge.selectors import CorrelationSelector

# Remove features with correlation > 0.85
corr_selector = CorrelationSelector(threshold=0.85)

X_uncorr = corr_selector.fit_transform(X)

print(f"Original features: {X.shape[1]}")
print(f"After removing correlated: {X_uncorr.shape[1]}")
print(f"Removed features: {set(X.columns) - set(X_uncorr.columns)}")

## Variance-Based Selection

Remove low-variance features:

In [ ]:
from forge.selectors import VarianceSelector

# Add some low-variance features to demonstrate
X_with_low_var = X.copy()
X_with_low_var['constant'] = 1.0  # Zero variance
X_with_low_var['low_var'] = np.random.choice([0, 1], size=len(X), p=[0.99, 0.01])  # Low variance

# Remove features with variance < 0.01
var_selector = VarianceSelector(threshold=0.01)

X_high_var = var_selector.fit_transform(X_with_low_var)

print(f"Original features: {X_with_low_var.shape[1]}")
print(f"After removing low variance: {X_high_var.shape[1]}")
print(f"Removed: {set(X_with_low_var.columns) - set(X_high_var.columns)}")

## SHAP-Based Selection

Use SHAP values for interpretable feature selection:

In [ ]:
# Note: Requires shap package
try:
    from forge.selectors import ShapSelector
    
    shap_selector = ShapSelector(k=10)
    X_shap = shap_selector.fit_transform(X, y)
    
    print(f"Selected {X_shap.shape[1]} features using SHAP:")
    print(list(X_shap.columns))
except ImportError:
    print("SHAP not installed. Install with: pip install forge-features[shap]")

## Combining Selection Methods

You can chain multiple selectors:

In [ ]:
from forge.transformers import ForgePipeline

# Create a selection pipeline
selection_pipeline = ForgePipeline([
    ('remove_low_var', VarianceSelector(threshold=0.01)),
    ('remove_correlated', CorrelationSelector(threshold=0.9)),
    ('select_top', ImportanceSelector(k=10)),
])

X_final = selection_pipeline.fit_transform(X_with_low_var, y)

print(f"Original: {X_with_low_var.shape[1]} features")
print(f"Final: {X_final.shape[1]} features")
print(f"Selected: {list(X_final.columns)}")

## Next Steps

- [04_sklearn_integration.ipynb](04_sklearn_integration.ipynb) - Use Forge with sklearn pipelines